# Comparing Run Time Across Use Cases and Activity-Model Treatments

**The situation:** usecases 01-03 each took *some* amount of wall-clock time
to run — a single-point `solve()` call in 01 and 02, a full 2000-step batch
simulation in 03. Both usecases 01 and 02 also showed that `use_activity=True`
(Davies correction) and reaching for PHREEQC instead of PyOMES's own solver
both change the *answer* slightly (Section 6 of each). This notebook asks a
different question: how much do those same choices change the *run time*?

This is a **benchmarking** notebook, not a scientific-results notebook — the
chemistry and parameters are copied verbatim from usecases 01-03 (same
recipe, same reaction networks, same batch-growth setup) so the workloads
being timed are the *actual* usecases already demonstrated, not toy
stand-ins. Wall-clock timing is inherently noisy (background load, CPU
frequency scaling, GC pauses), so every measurement below is a mean over
repeated calls after a fixed warmup, not a single stopwatch read.

**Two axes of comparison, and one caveat about mixing them:**

- **By usecase** — 01 (pH prediction), 02 (gas-liquid CO₂ equilibration), and
  03 (E. coli batch growth) represent three different *kinds* of workload:
  01 and 02 are a single equilibrium snapshot, while 03 is a 20-hour dynamic
  simulation stepped 2000 times, each step re-solving the same kind of
  equilibrium problem 01/02 solve *once*. So "01 vs 03 run time" is really
  "cost of one solve vs. cost of two thousand solves plus kinetics and
  transport" — an honest comparison, but not an apples-to-apples one, which
  is why the plots below keep those units labelled explicitly rather than
  implying they're the same thing.
- **By activity-model / engine treatment** — four combinations, matching
  usecase 01/02's own Section 6 exactly: **PyOMES ideal** (no activity
  correction), **PyOMES Davies** (`use_activity=True, activity_model="davies"`),
  **PHREEQC ideal-equivalent** (the `-gamma 1e6 0` trick from usecase 01
  §6c/usecase 02 §6, forcing γ→1), and **PHREEQC default**
  (`vitens.dat`'s WATEQ Debye-Hückel, via `PHREEQCChemicalEquilibriumEngine`).
  Usecase 03 only has the first two — there is no dynamic-`ControlVolume`
  PHREEQC path in PyOMES, so PHREEQC timing is only meaningful for the
  single-snapshot usecases.

PHREEQC timing needs the optional `phreeqpython` package
(`pip install PyOMES[phreeqc]`); if it isn't installed, this notebook skips
those cells and reports only the three PyOMES-only rows.

In [ ]:
import sys
import time
import warnings
import math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Run from inside the PyOMES repo")

sys.path.insert(0, str(_find_repo() / "models"))

from PyOMES.chemistry.species import Species
from PyOMES.chemistry.common_species import (
    H2O, H_plus, OH_minus,
    H3PO4, H2PO4_minus, HPO4_2minus, PO4_3minus,
    NH3, NH4_plus,
    CO2, HCO3_minus, CO3_2minus,
)
from PyOMES.chemistry import HenryEquilibrium
from PyOMES.reactions import (
    EquilibriumReaction, ReactionBuilder, ReactionSystem, StoichiometryEntry,
)
from PyOMES.chemical_equilibrium.nr_engine import NRChemicalEquilibriumEngine
from PyOMES.core import (
    ControlVolume, GasPhase, LiquidPhase,
    KineticTransferModel, EquilibriumTransferModel,
    Simulation, GasFeed, PressureReliefVent,
)
from PyOMES.core.phases import R_L_ATM_MOL_K

# Same self-correcting-solver warnings usecase 03 silences (AccuracyWarning /
# ConservationWarning at dt_h=0.01) -- silenced only so the timing cells'
# output stays readable, not because anything here is wrong.
from PyOMES.monitoring.accuracy import AccuracyWarning
from PyOMES.monitoring.conservation import ConservationWarning
warnings.filterwarnings("ignore", category=AccuracyWarning)
warnings.filterwarnings("ignore", category=ConservationWarning)

def _e(sp, coeff, phase="liquid"):
    return StoichiometryEntry(species=sp, phase=phase, coefficient=coeff)

try:
    import phreeqpython  # noqa: F401  -- actually probe for the optional dep;
    # PHREEQCChemicalEquilibriumEngine itself imports phreeqpython lazily
    # inside __init__, so importing only the wrapper class below would
    # succeed even without phreeqpython installed.
    from PyOMES.chemical_equilibrium.phreeqc_engine import PHREEQCChemicalEquilibriumEngine
    _HAVE_PHREEQC = True
    print("phreeqpython available - PHREEQC timing cells will run.")
except ImportError as exc:
    _HAVE_PHREEQC = False
    print(f"phreeqpython not installed ({exc}); skipping PHREEQC timing cells.")
    print("Install with: pip install PyOMES[phreeqc]")

# (usecase, engine/activity-model) -> seconds. For usecases 01-02 this is
# seconds per single engine.solve() call; for usecase 03 it's seconds for
# the *entire* 20 h / 2000-step batch run -- see the markdown notes below
# on why those two things aren't the same kind of quantity.
TIMINGS = {}

def time_repeated(fn, reps=100, warmup=3, trials=5):
    """Best-of-`trials` mean wall time per call, after a fixed warmup.

    Sub-millisecond NR solves are short enough that a *single* timed pass
    is dominated by OS scheduling noise, not the computation being
    measured -- the standard fix (same one `timeit.repeat` uses) is to
    repeat the whole timed pass several times and keep the fastest, since
    noise only ever makes a pass slower, never faster.

    The warmup matters more than usual here too: the *first* call into a
    freshly-built engine/solver pays one-time Python import and (for
    PHREEQC) IPC-connection costs that have nothing to do with the
    activity model being timed, and would otherwise bias whichever
    engine happens to run first in a given cell.
    """
    for _ in range(warmup):
        fn()
    best = None
    for _ in range(trials):
        t0 = time.perf_counter()
        for _ in range(reps):
            fn()
        t1 = time.perf_counter()
        mean = (t1 - t0) / reps
        best = mean if best is None else min(best, mean)
    return best

print("Imports OK")


## 1  Usecase 01 — pH prediction (single equilibrium solve)

Exactly [usecase 01](01_predict_ph_simple_liquid.ipynb)'s chemistry (the
five-reaction phosphate/ammonium network) and its M9-like recipe
(22 mmol/L KH₂PO₄, 18.7 mmol/L NH₄Cl), timed as repeated `engine.solve()`
calls at that one fixed point — the recipe doesn't matter for timing (NR
converges from the same starting guess every call), only that it's a
realistic, already-validated one rather than an arbitrary composition.

In [ ]:
water = EquilibriumReaction(
    stoichiometry=[_e(H2O, -1), _e(H_plus, +1), _e(OH_minus, +1)],
    log_K=-14.0, label="water",
)
p1 = EquilibriumReaction(
    stoichiometry=[_e(H3PO4, -1), _e(H2PO4_minus, +1), _e(H_plus, +1)],
    log_K=-2.15, total_id="H3PO4", label="p1",
)
p2 = EquilibriumReaction(
    stoichiometry=[_e(H2PO4_minus, -1), _e(HPO4_2minus, +1), _e(H_plus, +1)],
    log_K=-7.20, total_id="H3PO4", label="p2",
)
p3 = EquilibriumReaction(
    stoichiometry=[_e(HPO4_2minus, -1), _e(PO4_3minus, +1), _e(H_plus, +1)],
    log_K=-12.35, total_id="H3PO4", label="p3",
)
nh4 = EquilibriumReaction(
    stoichiometry=[_e(NH4_plus, -1), _e(NH3, +1), _e(H_plus, +1)],
    log_K=-9.25, total_id="NH3", label="nh4",
)

CT_P = 0.022   # mol/L KH2PO4 -> mol/L total phosphate, mol/L K+
CT_N = 0.0187  # mol/L NH4Cl  -> mol/L total ammoniacal N, mol/L Cl-

engine01_ideal = NRChemicalEquilibriumEngine.from_reactions([water, p1, p2, p3, nh4])
engine01_davies = NRChemicalEquilibriumEngine.from_reactions(
    [water, p1, p2, p3, nh4], use_activity=True, activity_model="davies",
)

def solve01(engine):
    return engine.solve(
        totals={"H3PO4": CT_P, "NH3": CT_N}, strong_ions={"CT_K": CT_P, "CT_Cl": CT_N},
    )

TIMINGS[("01 pH prediction", "PyOMES ideal")] = time_repeated(lambda: solve01(engine01_ideal))
TIMINGS[("01 pH prediction", "PyOMES Davies")] = time_repeated(lambda: solve01(engine01_davies))

print(f"01 PyOMES ideal:  {TIMINGS[('01 pH prediction', 'PyOMES ideal')]*1e3:.4f} ms/solve")
print(f"01 PyOMES Davies: {TIMINGS[('01 pH prediction', 'PyOMES Davies')]*1e3:.4f} ms/solve")


### PHREEQC, usecase 01

Reuses usecase 01 §6c's minimal ideal database (`-gamma 1e6 0` on every
charged species, forcing γ→1) for the PHREEQC-ideal-equivalent point, and
`PHREEQCChemicalEquilibriumEngine` (`vitens.dat`, WATEQ Debye-Hückel) for
the PHREEQC-default point — `use_warmstart=False` for the same reason
usecase 01 needed it: the default warmstart reuses the previous solution
*incrementally*, which would corrupt repeated identical solves.

In [ ]:
if _HAVE_PHREEQC:
    import tempfile
    from phreeqpython import PhreeqPython as _RawPhreeqPython

    _IDEAL_DB_01 = """\
SOLUTION_MASTER_SPECIES
H       H+      -1.     H       1.008
H(0)    H2      0.0     H
H(1)    H+      -1.     0.0
E       e-      0.0     0.0     0.0
O       H2O     0.0     O       16.00
O(0)    O2      0.0     O
O(-2)   H2O     0.0     0.0
P       PO4-3   0.0     P       30.974
N       NH4+    0.0     N       14.0067
K       K+      0.0     K       39.098
Cl      Cl-     0.0     Cl      35.453

SOLUTION_SPECIES
H+ = H+
        log_k           0.0
        -gamma          1e6     0
e- = e-
        log_k           0.0
H2O = H2O
        log_k           0.0
2 H+ + 2 e- = H2
        log_k           -3.15
2 H2O = O2 + 4 H+ + 4 e-
        log_k           -86.08
H2O = OH- + H+
        log_k           -14.0
        -gamma          1e6     0
PO4-3 = PO4-3
        log_k           0.0
        -gamma          1e6     0
PO4-3 + H+ = HPO4-2
        log_k           12.35
        -gamma          1e6     0
PO4-3 + 2H+ = H2PO4-
        log_k           19.55
        -gamma          1e6     0
PO4-3 + 3H+ = H3PO4
        log_k           21.70
NH4+ = NH4+
        log_k           0.0
        -gamma          1e6     0
NH4+ = NH3 + H+
        log_k           -9.25
K+ = K+
        log_k           0.0
        -gamma          1e6     0
Cl- = Cl-
        log_k           0.0
        -gamma          1e6     0
END
"""
    _db_dir = Path(tempfile.gettempdir())
    (_db_dir / "vlsim_ideal_phosphate_ammonium_runtime.dat").write_text(_IDEAL_DB_01)
    pp_ideal01 = _RawPhreeqPython(database="vlsim_ideal_phosphate_ammonium_runtime.dat",
                                   database_directory=_db_dir)

    def solve01_pq_ideal():
        sol = pp_ideal01.add_solution_raw({
            "P": CT_P * 1e3, "N": CT_N * 1e3, "K": CT_P * 1e3, "Cl": CT_N * 1e3,
            "temp": 25.0, "pH": "7 charge", "units": "mmol/L",
        })
        ph = sol.pH
        sol.forget()
        return ph

    engine01_pq_default = PHREEQCChemicalEquilibriumEngine(
        {"H3PO4": CT_P * 1e3, "NH3": CT_N * 1e3, "CT_K": CT_P * 1e3, "CT_Cl": CT_N * 1e3},
        component_map={"H3PO4": "P", "NH3": "N(-3)", "CT_K": "K", "CT_Cl": "Cl"},
        use_warmstart=False,
    )

    def solve01_pq_default():
        return engine01_pq_default.solve(
            totals={"H3PO4": CT_P, "NH3": CT_N, "CT_K": CT_P, "CT_Cl": CT_N},
        )

    TIMINGS[("01 pH prediction", "PHREEQC ideal (gamma->1)")] = time_repeated(solve01_pq_ideal, reps=10, warmup=2, trials=3)
    TIMINGS[("01 pH prediction", "PHREEQC default (WATEQ D-H)")] = time_repeated(solve01_pq_default, reps=10, warmup=2, trials=3)

    print(f"01 PHREEQC ideal:   {TIMINGS[('01 pH prediction', 'PHREEQC ideal (gamma->1)')]*1e3:.3f} ms/solve")
    print(f"01 PHREEQC default: {TIMINGS[('01 pH prediction', 'PHREEQC default (WATEQ D-H)')]*1e3:.3f} ms/solve")
else:
    print("Skipping PHREEQC timing for usecase 01 (phreeqpython not installed).")


## 2  Usecase 02 — gas-liquid CO₂ equilibration (single equilibrium solve)

[Usecase 02](02_equilibrate_with_atmospheric_gas.ipynb)'s network extends
usecase 01 with the two-step carbonate ladder, solved at the same M9-like
recipe plus dissolved CO₂ from a 400 ppm atmosphere (usecase 02 §2's
`kH_mol_L_atm` Henry conversion, reproduced here). Two more reactions than
usecase 01, so any per-reaction-network-size overhead in the NR tableau
build/solve should show up as a slightly larger 01-vs-02 gap within the
*same* activity-model column.

In [ ]:
co2_first = EquilibriumReaction(
    stoichiometry=[_e(CO2, -1), _e(H2O, -1), _e(HCO3_minus, +1), _e(H_plus, +1)],
    log_K=-6.35, total_id="CO2", label="co2_first",
)
co2_second = EquilibriumReaction(
    stoichiometry=[_e(HCO3_minus, -1), _e(CO3_2minus, +1), _e(H_plus, +1)],
    log_K=-10.33, total_id="CO2", label="co2_second",
)

def kH_mol_L_atm(H_ref, dlnH, T_K, T_ref=298.15):
    H_ref_mol_L_atm = (H_ref / 1000.0) * 101325.0
    return H_ref_mol_L_atm * math.exp(dlnH * (1.0 / T_K - 1.0 / T_ref))

CT_CO2_atm = kH_mol_L_atm(H_ref=3.4e-4, dlnH=2400.0, T_K=298.15) * 400e-6  # 400 ppm at 25 C

engine02_ideal = NRChemicalEquilibriumEngine.from_reactions(
    [water, p1, p2, p3, nh4, co2_first, co2_second]
)
engine02_davies = NRChemicalEquilibriumEngine.from_reactions(
    [water, p1, p2, p3, nh4, co2_first, co2_second],
    use_activity=True, activity_model="davies",
)

def solve02(engine):
    return engine.solve(
        totals={"H3PO4": CT_P, "NH3": CT_N, "CO2": CT_CO2_atm},
        strong_ions={"CT_K": CT_P, "CT_Cl": CT_N},
    )

TIMINGS[("02 gas-liquid CO2", "PyOMES ideal")] = time_repeated(lambda: solve02(engine02_ideal))
TIMINGS[("02 gas-liquid CO2", "PyOMES Davies")] = time_repeated(lambda: solve02(engine02_davies))

print(f"02 PyOMES ideal:  {TIMINGS[('02 gas-liquid CO2', 'PyOMES ideal')]*1e3:.4f} ms/solve")
print(f"02 PyOMES Davies: {TIMINGS[('02 gas-liquid CO2', 'PyOMES Davies')]*1e3:.4f} ms/solve")


### PHREEQC, usecase 02

Same idea as usecase 01, with usecase 02's CO₂-extended ideal database
(adds the `C` master species and carbonate reactions to the §6c database
above) for the PHREEQC-ideal-equivalent point.

In [ ]:
if _HAVE_PHREEQC:
    _IDEAL_DB_02 = """\
SOLUTION_MASTER_SPECIES
H       H+      -1.     H       1.008
H(0)    H2      0.0     H
H(1)    H+      -1.     0.0
E       e-      0.0     0.0     0.0
O       H2O     0.0     O       16.00
O(0)    O2      0.0     O
O(-2)   H2O     0.0     0.0
C       CO2     0.0     C       12.011
P       PO4-3   0.0     P       30.974
N       NH4+    0.0     N       14.0067
K       K+      0.0     K       39.098
Cl      Cl-     0.0     Cl      35.453

SOLUTION_SPECIES
H+ = H+
        log_k           0.0
        -gamma          1e6     0
e- = e-
        log_k           0.0
H2O = H2O
        log_k           0.0
2 H+ + 2 e- = H2
        log_k           -3.15
2 H2O = O2 + 4 H+ + 4 e-
        log_k           -86.08
H2O = OH- + H+
        log_k           -14.0
        -gamma          1e6     0
CO2 = CO2
        log_k           0.0
        -gamma          1e6     0
CO2 + H2O = HCO3- + H+
        log_k           -6.35
        -gamma          1e6     0
HCO3- = CO3-2 + H+
        log_k           -10.33
        -gamma          1e6     0
PO4-3 = PO4-3
        log_k           0.0
        -gamma          1e6     0
PO4-3 + H+ = HPO4-2
        log_k           12.35
        -gamma          1e6     0
PO4-3 + 2H+ = H2PO4-
        log_k           19.55
        -gamma          1e6     0
PO4-3 + 3H+ = H3PO4
        log_k           21.70
NH4+ = NH4+
        log_k           0.0
        -gamma          1e6     0
NH4+ = NH3 + H+
        log_k           -9.25
K+ = K+
        log_k           0.0
        -gamma          1e6     0
Cl- = Cl-
        log_k           0.0
        -gamma          1e6     0
END
"""
    _db_dir = Path(tempfile.gettempdir())
    (_db_dir / "vlsim_ideal_gas_liquid_runtime.dat").write_text(_IDEAL_DB_02)
    pp_ideal02 = _RawPhreeqPython(database="vlsim_ideal_gas_liquid_runtime.dat",
                                   database_directory=_db_dir)

    def solve02_pq_ideal():
        sol = pp_ideal02.add_solution_raw({
            "C": CT_CO2_atm * 1e3, "P": CT_P * 1e3, "N": CT_N * 1e3,
            "K": CT_P * 1e3, "Cl": CT_N * 1e3,
            "temp": 25.0, "pH": "7 charge", "units": "mmol/L",
        })
        ph = sol.pH
        sol.forget()
        return ph

    engine02_pq_default = PHREEQCChemicalEquilibriumEngine(
        {"H3PO4": CT_P * 1e3, "NH3": CT_N * 1e3, "CO2": 1.0,
         "CT_K": CT_P * 1e3, "CT_Cl": CT_N * 1e3},
        component_map={"H3PO4": "P", "NH3": "N(-3)", "CO2": "C",
                       "CT_K": "K", "CT_Cl": "Cl"},
        use_warmstart=False,
    )

    def solve02_pq_default():
        return engine02_pq_default.solve(
            totals={"H3PO4": CT_P, "NH3": CT_N, "CO2": CT_CO2_atm,
                    "CT_K": CT_P, "CT_Cl": CT_N},
        )

    TIMINGS[("02 gas-liquid CO2", "PHREEQC ideal (gamma->1)")] = time_repeated(solve02_pq_ideal, reps=10, warmup=2, trials=3)
    TIMINGS[("02 gas-liquid CO2", "PHREEQC default (WATEQ D-H)")] = time_repeated(solve02_pq_default, reps=10, warmup=2, trials=3)

    print(f"02 PHREEQC ideal:   {TIMINGS[('02 gas-liquid CO2', 'PHREEQC ideal (gamma->1)')]*1e3:.3f} ms/solve")
    print(f"02 PHREEQC default: {TIMINGS[('02 gas-liquid CO2', 'PHREEQC default (WATEQ D-H)')]*1e3:.3f} ms/solve")
else:
    print("Skipping PHREEQC timing for usecase 02 (phreeqpython not installed).")


## 3  Usecase 03 — E. coli batch growth (full dynamic simulation)

[Usecase 03](03_grow_ecoli_on_acetic_acid.ipynb)'s vessel, exactly as built
there: same species, same Monod parameters, same 2 L/37 °C/sparged-air
setup, same 20 h horizon at 2000 steps. The only new mechanic is
`ReactionSystem.configure_engine(use_activity=, activity_model=)`, called
*before* the first `.engine`/`.advance()` access — this is how the
activity treatment is chosen for a `ControlVolume`-driven system, the
dynamic-simulation equivalent of passing `use_activity=` directly to
`NRChemicalEquilibriumEngine.from_reactions()` in usecases 01-02.

A tiny throwaway run (5 steps) warms up each configuration before the
timed run, for the same reason `time_repeated` warms up above: the first
call into a freshly-built engine pays one-time import/JIT costs that
would otherwise unfairly penalise whichever configuration happens to run
first.

In [ ]:
ACETIC_ACID = Species(id="AceticAcid", atoms={"C": 2, "H": 4, "O": 2}, charge=0, MW=60.052)
ACETATE_MINUS = Species(id="Acetate-", atoms={"C": 2, "H": 3, "O": 2}, charge=-1, MW=59.044)
ECOLI = Species(id="Ecoli", atoms={"C": 1, "H": 1.8, "O": 0.5, "N": 0.2}, charge=0, MW=24.626)

acetate_eq = EquilibriumReaction(
    stoichiometry=[_e(ACETIC_ACID, -1), _e(ACETATE_MINUS, +1), _e(H_plus, +1)],
    log_K=-4.756, total_id="AceticAcid", label="acetate_eq",
)

mu_max_per_h, Ks_gL, Yxs, Ko2_gL = 0.30, 5e-3, 0.36, 0.2e-3
growth = ReactionBuilder.monod_aerobic_growth(
    substrate=ACETIC_ACID, biomass=ECOLI,
    mu_max_per_h=mu_max_per_h, Ks_gL=Ks_gL, yield_gX_gS=Yxs, Ko2_gL=Ko2_gL,
    balance="CHNO", label="growth_on_AceticAcid",
)

V_total_L, headspace_frac = 2.0, 0.20
V_liq = V_total_L * (1.0 - headspace_frac)
V_gas = V_total_L * headspace_frac
T_K = 310.15   # 37 C
n_total_gas = (1.0 * V_gas) / (R_L_ATM_MOL_K * T_K)

C_AcOH0 = 4.0 / ACETIC_ACID.MW    # 4 g/L acetic acid
X0 = 0.05 / ECOLI.MW              # 0.05 g/L inoculum

transfer_models = {
    "O2": KineticTransferModel(partition_model=HenryEquilibrium(H_ref=1.3e-5, dlnH=1500.0), k_transfer=150.0),
    "CO2": KineticTransferModel(partition_model=HenryEquilibrium(H_ref=3.4e-4, dlnH=2400.0), k_transfer=150.0 * 0.9),
    "N2": EquilibriumTransferModel(HenryEquilibrium(H_ref=6.4e-6, dlnH=1300.0)),
}

def build_usecase03_cv(use_activity, activity_model):
    system = ReactionSystem(
        [water, p1, p2, p3, nh4, co2_first, co2_second, acetate_eq, growth],
        label="ecoli_on_acetate", solver="newton_raphson",
    )
    system.configure_engine(use_activity=use_activity, activity_model=activity_model)

    gas_phase = GasPhase(
        n_mol={"O2": n_total_gas * 0.2095, "N2": n_total_gas * 0.7901, "CO2": n_total_gas * 0.0004},
        V_L=V_gas, T_K=T_K,
    )
    liquid_phase = LiquidPhase(
        n_mol={
            "H3PO4": CT_P * V_liq, "NH3": CT_N * V_liq,
            "AceticAcid": C_AcOH0 * V_liq, "Ecoli": X0 * V_liq,
            "K+": CT_P * V_liq, "Cl-": CT_N * V_liq,
            "O2": 0.0, "CO2": 0.0, "N2": 0.0,
        },
        V_L=V_liq, T_K=T_K,
    )
    gas_feed = GasFeed(
        vvm_min=1.0, y={"O2": 0.2095, "N2": 0.7901, "CO2": 0.0004}, P_inlet_atm=1.0,
        phase_key="gas", liquid_phase_key="liquid", label="gas_feed",
    )
    vent = PressureReliefVent(P_set_atm=1.0, mode="instant")
    return ControlVolume(
        phases={"gas": gas_phase, "liquid": liquid_phase},
        transfer_models=transfer_models,
        boundaries=[gas_feed, vent],
        reaction_system=system,
        label="batch_ecoli",
    )

def run_usecase03(use_activity, activity_model, tau_h, n_steps):
    cv = build_usecase03_cv(use_activity, activity_model)
    sim = Simulation(cvs={"main": cv}, label="runtime_compare")
    return sim.run(tau_h=tau_h, n_steps=n_steps)

# Warmup: tiny throwaway runs, discarded.
run_usecase03(False, "davies", tau_h=0.05, n_steps=5)
run_usecase03(True, "davies", tau_h=0.05, n_steps=5)

# Timed runs: usecase 03's actual parameters (20 h, 2000 steps).
t0 = time.perf_counter()
result03_ideal = run_usecase03(False, "davies", tau_h=20.0, n_steps=2000)
TIMINGS[("03 E. coli batch growth", "PyOMES ideal")] = time.perf_counter() - t0

t0 = time.perf_counter()
result03_davies = run_usecase03(True, "davies", tau_h=20.0, n_steps=2000)
TIMINGS[("03 E. coli batch growth", "PyOMES Davies")] = time.perf_counter() - t0

print(f"03 PyOMES ideal:  {TIMINGS[('03 E. coli batch growth', 'PyOMES ideal')]:.3f} s for the full 20 h / 2000-step run")
print(f"03 PyOMES Davies: {TIMINGS[('03 E. coli batch growth', 'PyOMES Davies')]:.3f} s for the full 20 h / 2000-step run")
print("PHREEQC: no dynamic-ControlVolume path exists in PyOMES, so there is no PHREEQC timing for usecase 03.")


## 4  All nine measurements, side by side

Printed before plotting so every number below is also available as plain
text (bar charts are for comparing shapes at a glance; exact values belong
in a table).

In [ ]:
UC_KEYS = ["01 pH prediction", "02 gas-liquid CO2", "03 E. coli batch growth"]
ENGINES = ["PyOMES ideal", "PyOMES Davies", "PHREEQC ideal (gamma->1)", "PHREEQC default (WATEQ D-H)"]

print(f"{'Usecase':<26}{'Engine / activity model':<30}{'Time':>18}")
print("-" * 74)
for uc in UC_KEYS:
    per_solve = uc.startswith(("01", "02"))
    for eng in ENGINES:
        val = TIMINGS.get((uc, eng))
        if val is None:
            continue
        disp = f"{val*1e3:.4f} ms/solve" if per_solve else f"{val:.3f} s (full run)"
        print(f"{uc:<26}{eng:<30}{disp:>18}")


## 5  Plotting the comparison

Two panels, both log-scale (the range here spans sub-millisecond NR solves
up to multi-second batch runs — a linear axis would flatten everything but
the largest bars to invisible slivers):

- **Left — absolute run time.** One group of bars per usecase, one colour
  per engine/activity-model treatment. Usecase 03 only shows two bars
  (PyOMES ideal/Davies) since PHREEQC has no entry there.
- **Right — time relative to that usecase's own PyOMES-ideal run**, i.e.
  each usecase's own ideal bar is normalised to 1.0. This isolates the
  *marginal* cost of Davies/PHREEQC from the much bigger difference in
  absolute magnitude between "one solve" (01, 02) and "2000 solves plus
  kinetics and transport" (03) that dominates the left panel — the
  question the right panel answers is "how much overhead does this
  treatment add, given whatever usecase it's used in," independent of
  which usecase that is.

In [ ]:
UC_LABELS = [
    "01  pH prediction\n(1 equilibrium solve)",
    "02  Gas-liquid CO2\n(1 equilibrium solve)",
    "03  E. coli batch growth\n(full 20h / 2000-step run)",
]
COLORS = {
    "PyOMES ideal":                  "#2a78d6",
    "PyOMES Davies":                 "#eb6834",
    "PHREEQC ideal (gamma->1)":     "#1baf7a",
    "PHREEQC default (WATEQ D-H)":  "#eda100",
}

n_uc = len(UC_KEYS)
bar_w, bar_gap = 0.20, 0.03
x = np.arange(n_uc)

# Usecase 03 only ever has 2 of the 4 engine columns (no PHREEQC path for a
# dynamic ControlVolume) -- rather than reserve 4 fixed slot positions and
# leave two visibly empty, each usecase group is centered on however many
# engines actually apply to *it*, so every group's bars sit under its own
# tick regardless of how many columns that usecase has.
def _grouped_bars(ax, values_by_engine, collect_legend):
    handles, labels = [], []
    for i, uc in enumerate(UC_KEYS):
        present = [(eng, values_by_engine[eng][i]) for eng in ENGINES
                   if values_by_engine[eng][i] is not None]
        n_present = len(present)
        for k, (eng, val) in enumerate(present):
            offset = (k - (n_present - 1) / 2) * (bar_w + bar_gap)
            bar = ax.bar(x[i] + offset, val, width=bar_w, color=COLORS[eng], edgecolor="none")
            if collect_legend and eng not in labels:
                handles.append(bar[0])
                labels.append(eng)
    ax.set_xticks(x)
    ax.set_xticklabels(UC_LABELS, fontsize=8.5)
    ax.set_yscale("log")
    ax.grid(True, which="major", axis="y", alpha=0.3, linewidth=0.8)
    ax.grid(True, which="minor", axis="y", alpha=0.12, linewidth=0.6)
    return handles, labels

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
fig.subplots_adjust(bottom=0.26, wspace=0.3)

abs_vals = {eng: [TIMINGS.get((uc, eng)) for uc in UC_KEYS] for eng in ENGINES}
handles, labels = _grouped_bars(ax1, abs_vals, collect_legend=True)
ax1.set_ylabel("wall-clock time (s, log scale)")
ax1.set_title("Absolute run time")

rel_vals = {}
for eng in ENGINES:
    row = []
    for uc in UC_KEYS:
        val, base = TIMINGS.get((uc, eng)), TIMINGS.get((uc, "PyOMES ideal"))
        row.append(val / base if (val is not None and base is not None) else None)
    rel_vals[eng] = row
_grouped_bars(ax2, rel_vals, collect_legend=False)
ax2.axhline(1.0, color="#c3c2b7", lw=1, zorder=0)
ax2.set_ylabel("time relative to that usecase's PyOMES-ideal run (log scale)")
ax2.set_title("Overhead relative to PyOMES ideal")

fig.legend(handles, labels, loc="lower center", ncol=4, fontsize=8.5, frameon=False,
           bbox_to_anchor=(0.5, 0.02))
plt.show()


## Takeaways

- **Davies vs. ideal is cheap.** Across usecases 01 and 02, adding the
  activity-coefficient correction costs roughly the same small constant
  factor per solve — noticeable in the right panel, invisible next to
  usecase 03's total in the left one.
- **PHREEQC's overhead is a completely different order of magnitude.**
  Every PHREEQC call round-trips through `phreeqpython`'s IPC layer to a
  separate PHREEQC process, which costs far more than the NR solve itself
  — the reason usecases 01/02 use it only for periodic validation (a
  handful of points, Section 6 of each), never as the per-step solver a
  dynamic simulation like usecase 03 would need.
- **Usecase-to-usecase differences dwarf activity-model differences.**
  The single biggest lever on run time here isn't which activity model is
  used at all — it's whether the workload is "solve once" or "solve two
  thousand times inside a stepped simulation." That's a completely
  different kind of scaling question (number of steps, step size,
  solver warmstart effectiveness) from anything Section 5's right panel
  addresses, and outside this notebook's scope.

## Where to go next

- **Why Davies and PHREEQC give slightly different answers, not just
  different run times** — [usecase 01](01_predict_ph_simple_liquid.ipynb)
  §6 and [usecase 02](02_equilibrate_with_atmospheric_gas.ipynb) §6 walk
  through the accuracy side of this same comparison.
- **Where usecase 03's own per-step cost actually goes** (NR solve vs.
  kinetics vs. transport vs. bookkeeping) — a per-step profiling breakdown
  is out of scope here but would be the natural next cut if usecase 03's
  run time itself, not just its activity-model sensitivity, becomes the
  question.